In [1]:
import os
import re
import json
import multiprocessing as mp
import pandas as pd


## PaddleOCR

[PaddleOCR](https://github.com/PaddlePaddle/PaddleOCR) is an open-source optical character recognition (OCR) toolkit. Given an image, it locates regions of text and transcribes them into machine-readable strings. It was chosen for this pipeline after comparing it against several alternatives (see `README.md`): it handles handwritten and typed text reasonably well, runs entirely locally (no API cost or internet dependency), and integrates cleanly with Python's `multiprocessing`.

The function that actually calls PaddleOCR, `extract_page()`, lives in **`ocr_worker.py`** (a separate file next to this notebook) rather than in a cell here. On macOS, `multiprocessing`'s default "spawn" start method launches fresh worker processes that re-`import` whatever module the target function was defined in; a Jupyter kernel's `__main__` isn't a real importable module, so a function defined in a notebook cell can't be found by those workers (`AttributeError: Can't get attribute 'extract_page' on <module '__main__' (built-in)>`). Defining it in its own `.py` file sidesteps that.

For a single image, `extract_page()`:

1. Initializes a `PaddleOCR` reader (`use_angle_cls=True` lets it correct for rotated text; `lang='en'` selects the English text-recognition model).
2. Runs `ocr.predict(path)` on the image, which returns a list of results containing the recognized text strings (`rec_texts`) **and, for each one, a recognition confidence score (`rec_scores`, 0–1)** — PaddleOCR's own estimate of how sure it is about that transcription.
3. Pairs each text string with its confidence score and returns them tagged with the image's filename, so results can be matched back to the manifest afterward.

Because each worker process in the multiprocessing pool needs its own PaddleOCR instance, the reader is initialized *inside* `extract_page()` rather than once at the top of the module.

The confidence scores carry through to the final output so that low-confidence transcriptions — PaddleOCR effectively "guessing" — are flagged rather than presented as equally reliable as everything else. See `LOW_CONFIDENCE_THRESHOLD` below.

In [2]:
from ocr_worker import extract_page


/Users/rfreedma/anaconda3/envs/lang/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/rfreedma/anaconda3/envs/lang/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Source images and manifest

The manifest here is a CSV published from a Google Sheet, listing every source image alongside the archival **accession_id** it belongs to:

| column         | description                                                            |
|----------------|--------------------------------------------------------------------------|
| `filename`     | the image's filename, as it appears in `IMAGE_FOLDER`                  |
| `accession_id` | the archival accession this image belongs to                           |

Unlike a one-file-per-document manifest, **multiple images can share the same `accession_id`** — e.g. a recto/verso pair, or every page of a multi-page item. Each image's filename also embeds a two-digit sequence number (`..._id__01__...`, `..._id__02__...`) indicating its order within that accession's batch.

The pipeline groups images by `accession_id`, OCRs each image, reassembles each accession's pages in sequence order, and writes one JSON record per accession — with `accession_id` and the source `filenames` carried along as metadata — to a combined JSONL output file. Each line of recognized text also carries PaddleOCR's confidence score, and lines below `LOW_CONFIDENCE_THRESHOLD` are flagged so downstream consumers know where PaddleOCR was effectively guessing. This shape is meant to feed downstream structured-data extraction (e.g. a Pydantic + LLM step pulling out composer/piece entities) and a RAG pipeline, where each accession is the natural retrievable unit.

In [3]:
#Published Google Sheets CSV manifest (filename, accession_id columns)
MANIFEST_URL = "https://docs.google.com/spreadsheets/d/e/2PACX-1vSeA9kskfAurok_dyzdDf-pIFAEBtdnc7JdUnGxUO7HnxMTF8a3le7Z1uSMfgZWKQCR2SyvrxEcIkLi/pub?output=csv"
#Folder containing the source JPG images. These live in the sibling Melbourne Jewish Museum archive repo, matching the manifest's filename column
IMAGE_FOLDER = "../Melbourne_Jewish_Museum_Concert_Archive-main/images"
#Combined JSONL output: one record per accession_id
OUTPUT_PATH = "output/extracted_text.jsonl"
#Recognized lines with a PaddleOCR confidence score below this are flagged as low-confidence ("guesses")
LOW_CONFIDENCE_THRESHOLD = 0.80


In [4]:
#dtype=str keeps accession_id (e.g. "13795.3", "3921") from being parsed as a float and reformatted
manifest = pd.read_csv(MANIFEST_URL, dtype=str)
manifest.head()


,filename,accession_id
0,13795.3__01__13795-3_lowres584.jpg,13795.3
1,13810.12__01__13810.12.JPG,13810.12
2,3092__01__3092_lowres814.jpg,3092
3,3095__01__3095_lowres817.jpg,3095
4,3129__01__3129_lowres834.jpg,3129


## Running the pipeline

1. Check which manifest images are actually present in `IMAGE_FOLDER`, warning on anything missing.
2. Run `extract_page()` over all present images in parallel with a `multiprocessing.Pool`.
3. Group the results by `accession_id`, sort each accession's pages using the sequence number embedded in its filenames, flag any line below `LOW_CONFIDENCE_THRESHOLD`, and join the pages into a single `full_text`.
4. Write one JSON record per accession to `OUTPUT_PATH` (one record per line, JSONL), and print a summary of accessions containing low-confidence lines.

In [5]:
def image_path(filename):
    return os.path.join(IMAGE_FOLDER, filename)

def parse_seq(filename):
    #Pulls the "__NN__" sequence number out of a manifest filename, e.g. "3129__02__3129_verso.jpg" -> 2
    match = re.search(r"__(\d+)__", filename)
    return int(match.group(1)) if match else 0

missing = [f for f in manifest["filename"] if not os.path.exists(image_path(f))]
for f in missing:
    print(f"Warning: {f} not found in {IMAGE_FOLDER}/, skipping.")

to_process = manifest[~manifest["filename"].isin(missing)]
print(f"\nOCRing {len(to_process)} of {len(manifest)} manifest image(s).")



OCRing 194 of 194 manifest image(s).


In [6]:
with mp.Pool(processes=min(4, os.cpu_count())) as pool:
    ocr_results = pool.map(extract_page, [image_path(f) for f in to_process["filename"]])

lines_by_filename = {r["filename"]: r["lines"] for r in ocr_results}


/Users/rfreedma/anaconda3/envs/lang/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/rfreedma/anaconda3/envs/lang/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/rfreedma/anaconda3/envs/lang/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/rfreedma/anaconda3/envs/lang/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/rfreedma/anaconda3/envs/lang/lib/python3.11/site-packages/paddle/

In [7]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

flagged_accessions = []

with open(OUTPUT_PATH, "w") as out:
    for accession_id, group in to_process.groupby("accession_id"):
        pages = []
        for filename in group["filename"]:
            #Flags any recognized line PaddleOCR wasn't confident about
            lines = [
                {**line, "low_confidence": line["confidence"] < LOW_CONFIDENCE_THRESHOLD}
                for line in lines_by_filename[filename]
            ]
            pages.append({"filename": filename, "seq": parse_seq(filename), "lines": lines})
        pages.sort(key=lambda p: p["seq"])

        low_confidence_count = sum(l["low_confidence"] for p in pages for l in p["lines"])
        if low_confidence_count:
            flagged_accessions.append((accession_id, low_confidence_count))

        record = {
            "accession_id": str(accession_id),
            "filenames": [p["filename"] for p in pages],
            "pages": pages,
            "full_text": "\n".join(" ".join(l["text"] for l in p["lines"]) for p in pages),
            "low_confidence_line_count": low_confidence_count,
        }
        out.write(json.dumps(record) + "\n")

print(f"Wrote {to_process['accession_id'].nunique()} accession record(s) to {OUTPUT_PATH}")

if flagged_accessions:
    print(f"\n{len(flagged_accessions)} accession(s) contain lines below the confidence threshold ({LOW_CONFIDENCE_THRESHOLD}):")
    for accession_id, count in flagged_accessions:
        print(f"  - {accession_id}: {count} low-confidence line(s)")
else:
    print(f"\nNo lines fell below the confidence threshold ({LOW_CONFIDENCE_THRESHOLD}).")


Wrote 70 accession record(s) to output/extracted_text.jsonl

43 accession(s) contain lines below the confidence threshold (0.8):
  - 13795.3: 1 low-confidence line(s)
  - 3095: 1 low-confidence line(s)
  - 3129: 16 low-confidence line(s)
  - 3257: 1 low-confidence line(s)
  - 3287: 1 low-confidence line(s)
  - 3315.1: 9 low-confidence line(s)
  - 3315.2: 9 low-confidence line(s)
  - 3315.3: 9 low-confidence line(s)
  - 3362: 2 low-confidence line(s)
  - 3372: 1 low-confidence line(s)
  - 3373: 1 low-confidence line(s)
  - 3376: 1 low-confidence line(s)
  - 3439: 4 low-confidence line(s)
  - 3458: 2 low-confidence line(s)
  - 3619: 14 low-confidence line(s)
  - 3670: 9 low-confidence line(s)
  - 3681.3: 2 low-confidence line(s)
  - 3683.5: 5 low-confidence line(s)
  - 3685.55: 63 low-confidence line(s)
  - 3744: 2 low-confidence line(s)
  - 3777.2: 4 low-confidence line(s)
  - 3840: 1 low-confidence line(s)
  - 3921: 38 low-confidence line(s)
  - 3954: 3 low-confidence line(s)
  - 4023.

## Reviewing low-confidence transcriptions

`OUTPUT_PATH` is JSON Lines (JSONL): one self-contained JSON object per line, rather than one big JSON array. `pd.read_json(..., lines=True)` reads it straight into a DataFrame — one row per accession, with `pages` (and each page's `lines`) still nested as Python lists/dicts inside their cells.

`dtype={"accession_id": str}` matters here for the same reason it did when loading the manifest: without it, pandas infers `accession_id` as a float and mangles IDs like `"13795.3"`.

We'll load it two ways: first at the accession level (to see which accessions have the most low-confidence lines), then exploded down to one row per recognized line (to see the actual flagged text, sorted worst-confidence-first) — that's the table worth scrolling through before trusting the OCR output.

In [8]:
review_df = pd.read_json(OUTPUT_PATH, lines=True, dtype={"accession_id": str})

print(f"{len(review_df)} accessions, {review_df['low_confidence_line_count'].gt(0).sum()} with at least one low-confidence line.")
review_df.sort_values("low_confidence_line_count", ascending=False).head(10)[
    ["accession_id", "filenames", "low_confidence_line_count"]
]


70 accessions, 43 with at least one low-confidence line.


,accession_id,filenames,low_confidence_line_count
42,4332,"[4332__01__04332.000.000.001.dimgemu.jpg, 4332...",358
24,3685.55,"[3685.55__01__3685-1_lowres.jpg, 3685.55__02__...",63
51,4483,"[4483__01__4483_lowres350.jpg, 4483__02__4483b...",43
31,3921,[3921__01__3921.jpg],38
53,4485.1,"[4485.1__01__4485-1_lowres358.jpg, 4485.1__02_...",35
54,4486,"[4486__01__4486_lowres361.jpg, 4486__02__4486_...",35
4,3129,"[3129__01__3129_lowres834.jpg, 3129__02__3129_...",16
19,3619,"[3619__01__3619a_lowres169.jpg, 3619__02__3619...",14
48,4468,"[4468__01__04468.000.000.001.dimgemu.jpg, 4468...",14
21,3670,[3670__01__3670_lowres753.jpg],9


In [9]:
#Flatten to one row per recognized line, so individual low-confidence transcriptions can be inspected directly
lines_df = pd.DataFrame([
    {
        "accession_id": row["accession_id"],
        "filename": page["filename"],
        "seq": page["seq"],
        "text": line["text"],
        "confidence": line["confidence"],
        "low_confidence": line["low_confidence"],
    }
    for _, row in review_df.iterrows()
    for page in row["pages"]
    for line in page["lines"]
])

low_confidence_lines = lines_df[lines_df["low_confidence"]].sort_values("confidence")
print(f"{len(low_confidence_lines)} of {len(lines_df)} recognized lines are below the {LOW_CONFIDENCE_THRESHOLD} confidence threshold.")
low_confidence_lines


725 of 8143 recognized lines are below the 0.8 confidence threshold.


,accession_id,filename,seq,text,confidence,low_confidence
8096,7039,7039__01__7039_lowres580.jpg,1,,0.0000,True
6593,4485.1,4485.1__02__4485-1_verso_lowres359.jpg,2,,0.0000,True
5515,4332,4332__09__04332.000.000.009.dimgemu.jpg,9,,0.0000,True
8091,7012,7012__02__7012_p4.jpg,2,,0.0000,True
5662,4332,4332__10__04332.000.000.010.dimgemu.jpg,10,,0.0000,True
...,...,...,...,...,...,...
6567,4485.1,4485.1__01__4485-1_lowres358.jpg,1,ish ble dich erinessh dics Berard,0.7975,True
5938,4332,4332__11__04332.000.000.011.dimgemu.jpg,11,To,0.7988,True
5183,4332,4332__04__04332.000.000.004.dimgemu.jpg,4,(mooialite dostor involved in tho Profuno ease...,0.7989,True
2783,3685.55,3685.55__29__3685-29_lowres799.jpg,29,0z:,0.7993,True
